# Day 10 — FAISS index for corpus-wide dense search

Goal: add a FAISS index over the 28,541 cached bge-small-en-v1.5 chunk embeddings for **corpus-wide** dense search
(needed later for the cross-paper mode), and benchmark it against the current numpy search.

Plan, one step at a time:
1. Setup (Drive, packages, today's code).
2. Gate: the frozen within-paper pipeline still reproduces **R@5 = 0.795** on test.
3. A small look at FAISS vs numpy on one question.
4. Full benchmark: FAISS `IndexFlatIP` must give the same rankings/recall as numpy; build time, latency; HNSW grid (speed vs recall).
5. Corpus-wide `retrieve(..., paper_id=None)` through the full pipeline.

No LLM calls today, so no API keys are needed. Use a GPU runtime (query encoding + reranker).

In [4]:
# WHAT: Mounts Google Drive, finds the research-engine folder and makes it the working directory.
# WHY:  Every command below runs from the project root (data/, cache/ and the code live there).
#       glob + os.chdir is used because `%cd path  # comment` breaks in Colab.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
PROJECT = glob.glob('/content/drive/MyDrive/research-engine')[0]
os.chdir(PROJECT)
print(os.getcwd())
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/research-engine
 agent				 eval
 app				 experiments
 cache				 index
 configs			 ingest
 data				 README.md
 Day01-02.ipynb			 README_old.md
 Day03_hybrid_retrieval.ipynb	 requirements_old.txt
 Day04_rerank.ipynb		 requirements.txt
 Day05-06_refactor_agent.ipynb	'research-engine-day6 (1) (1).zip'
 Day07_agent_v2.ipynb		 results
 Day09_gemini_judge.ipynb	 retrieval


In [6]:
# WHAT: Installs the project's packages, now including faiss-cpu.
# WHY:  A fresh Colab session doesn't have rank_bm25, sentence-transformers' extras or faiss.
#       (Run this cell again after every runtime restart.)
!pip install -q -r requirements.txt
import faiss
print("faiss", faiss.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 89.9 MB/s eta 0:00:00
faiss 1.15.1


In [7]:
# WHAT: Checks that the data and the cached chunk embeddings are where the code expects them, and that a GPU is on.
# WHY:  The benchmark reuses cache/emb_BAAI_bge-small-en-v1.5_28541.npy (no re-embedding). If it's missing,
#       DenseIndex would silently re-encode all 28.5k chunks first (a few minutes on a GPU).
!ls -la data/processed/ cache/ | head -30
!nvidia-smi --query-gpu=name,memory.total --format=csv

cache/:
total 42969
drwx------  2 root root     4096 Oct 12 21:03 .
drwx------ 14 root root     4096 Oct 15 09:45 ..
-rw-------  1 root root 43839104 Oct 12 21:07 emb_BAAI_bge-small-en-v1.5_28541.npy
-rw-------  1 root root       41 Oct 12 22:47 rerank_BAAI_bge-reranker-v2-m3_dev_paper_top30_n887.json
-rw-------  1 root root    60548 Oct 12 22:47 rerank_BAAI_bge-reranker-v2-m3_dev_paper_top30_n887.npy
-rw-------  1 root root       42 Oct 12 22:59 rerank_BAAI_bge-reranker-v2-m3_test_paper_top30_n1309.json
-rw-------  1 root root    90392 Oct 12 22:59 rerank_BAAI_bge-reranker-v2-m3_test_paper_top30_n1309.npy

data/processed/:
total 22403
drwx------ 2 root root     4096 Oct 12 05:02 .
drwx------ 3 root root     4096 Oct 12 05:02 ..
-rw------- 1 root root 19801283 Oct 12 19:58 chunks.parquet
-rw------- 1 root root        0 Oct 11 23:23 .gitkeep
-rw------- 1 root root   913374 Oct 12 19:58 papers.parquet
-rw------- 1 root root  2216671 Oct 12 19:58 qasper_questions.jsonl
name, memory.total 

## Step 1 — gate: the frozen pipeline is unchanged
Within-paper search never touches FAISS (a paper has ~18 chunks), so this must print **REPRODUCED** with the same
numbers as Day 4/5. Reranker scores come from the cache, so it takes about a minute.

In [9]:
# WHAT: Re-runs the held-out retrieval evaluation (test split, within-paper scope) with today's code.
# WHY:  Proves the FAISS change didn't move the frozen result: final R@5 must be 0.795 (R@10 0.941, hit@1 0.522,
#       MRR 0.677). If it says MISMATCH, stop here.
!python -m eval.run_retrieval_eval --split test

28541 chunks | 1585 papers | 1309 questions with gold evidence (split=test, scope=paper)
Loading weights: 100% 199/199 [00:00<00:00, 4867.14it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
Loading cached reranker scores: /content/drive/MyDrive/research-engine/cache/rerank_BAAI_bge-reranker-v2-m3_test_paper_top30_n1309.npy

scope=paper  split=test  n=1309  reranker=BAAI/bge-reranker-v2-m3  excluded=['abstract']
| system | hit@1 | hit@5 | ev_recall@5 | ev_recall@10 | mrr@10 |
|---|---|---|---|---|---|
| bm25 | 0.254 | 0.735 | 0.612 | 0.849 | 0.455 |
| dense | 0.409 | 0.824 | 0.699 | 0.890 | 0.581 |
| hybrid_rrf | 0.370 | 0.814 | 0.692 | 0.896 | 0.557 |
| hybrid+rerank | 0.390 | 0.869 | 0.749 | 0.932 | 0.588 |
| final | 0.522 | 0.906 | 0.795 | 0.941 | 0.677 |

Rerank cost: 22566 pairs, 262 ms/question (from cache)
Results log: final ev_recall@5 = 0.795 | this run: 0.795 -> REPRODUCED
Saved /content/drive/MyDrive/research-en

## Step 2 — FAISS vs numpy on one question

In [10]:
# WHAT: Loads the cached embeddings, builds a FAISS IndexFlatIP and runs one question both ways.
# WHY:  To see what the index does before benchmarking it. The embeddings are unit length, so inner product
#       = cosine similarity: numpy `emb @ q` and IndexFlatIP should give the same top chunks and scores.
import numpy as np, time
from retrieval.config import RetrievalConfig
from retrieval.data import load_chunks
from retrieval.dense import DenseIndex
from retrieval.faiss_index import FaissIndex

chunks = load_chunks()
cfg = RetrievalConfig()
dense = DenseIndex(chunks["text"].tolist(), cfg.dense_model)          # cached embeddings, numpy backend
norms = np.linalg.norm(dense.emb, axis=1)
print("embeddings:", dense.emb.shape, f"| vector lengths {norms.min():.4f}-{norms.max():.4f} (should be 1)")

flat = FaissIndex(dense.emb, kind="flat")

q = "Which datasets are used for sentiment classification?"
qv = dense.encode_query(q)

t0 = time.perf_counter(); s = dense.emb @ qv; np_top = np.argsort(-s)[:5]; t_np = time.perf_counter() - t0
t0 = time.perf_counter(); f_sc, f_ix = flat.search(qv, 5); t_f = time.perf_counter() - t0

print(f"\nnumpy {1000*t_np:.2f} ms | FAISS flat {1000*t_f:.2f} ms  (one query; the benchmark times hundreds)")
for r, (a, b, fs) in enumerate(zip(np_top, f_ix[0], f_sc[0]), 1):
    print(f"{r}. numpy {chunks.chunk_id[a]:<28} {s[a]:.4f} | faiss {chunks.chunk_id[b]:<28} {fs:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
embeddings: (28541, 384) | vector lengths 1.0000-1.0000 (should be 1)
FAISS IndexFlatIP built in 0.03 s (28541 vectors, dim 384)

numpy 5.46 ms | FAISS flat 18.55 ms  (one query; the benchmark times hundreds)
1. numpy 1908.09590::015              0.8506 | faiss 1908.09590::015              0.8506
2. numpy 1602.07563::029              0.8356 | faiss 1602.07563::029              0.8356
3. numpy 2004.02214::015              0.8286 | faiss 2004.02214::015              0.8286
4. numpy 1812.06705::010              0.8286 | faiss 1812.06705::010              0.8286
5. numpy 1605.05195::006              0.8285 | faiss 1605.05195::006              0.8285


## Step 3 — full benchmark
For every dev (887) and test (1,309) question, encoded once with the pipeline's own `encode_query`:
* **Flat vs numpy:** same top-10/50/100 order and set? Largest score gap where they differ (should be ~1e-7 float
  rounding, not a real change). Gold evidence metrics must be equal. Dev numpy should also match the Day 3
  corpus dense numbers (R@5 0.1098, R@10 0.155).
* **Cost:** build time, index size, search latency (median/p95 per single query; batched per query).
* **HNSW:** M = 16, 32 × efSearch = 16…256: ANN recall@k vs exact search, gold metrics, latency.

In [11]:
# WHAT: Runs the FAISS benchmark (dev + test, default HNSW grid) and saves results/faiss_benchmark.json/.md.
# WHY:  This is today's measurement: is FAISS flat exact (it should be), and how much speed does HNSW buy at
#       what recall? Query vectors are cached in cache/qemb_*.npy, so a rerun skips the encoding.
#       Takes a few minutes; most of it is building the two HNSW graphs and the single-query timing loops.
!python -m eval.benchmark_faiss

Loading weights: 100% 199/199 [00:00<00:00, 10235.15it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
28541 chunk vectors, dim 384 | faiss 1.15.1 | 2 CPUs, FAISS threads 2
FAISS IndexFlatIP built in 0.03 s (28541 vectors, dim 384)
FAISS IndexHNSWFlat(M=16, efConstruction=200, efSearch=128) built in 9.29 s (28541 vectors, dim 384)
FAISS IndexHNSWFlat(M=32, efConstruction=200, efSearch=128) built in 10.41 s (28541 vectors, dim 384)

=== dev: 887 questions | query encoding 10.6 ms/q
  HNSW M=16  ef=16   ANN@10 0.7823  R@5 0.0995  0.099 ms/q
  HNSW M=16  ef=32   ANN@10 0.8786  R@5 0.1021  0.138 ms/q
  HNSW M=16  ef=64   ANN@10 0.9408  R@5 0.1047  0.214 ms/q
  HNSW M=16  ef=128  ANN@10 0.9752  R@5 0.1070  0.354 ms/q
  HNSW M=16  ef=256  ANN@10 0.9911  R@5 0.1087  0.593 ms/q
  HNSW M=32  ef=16   ANN@10 0.8101  R@5 0.0983  0.119 ms/q
  HNSW M=32  ef=32   ANN@10 0.9023  R@5 0.1105  0.167 ms/q
  HNSW M=32  ef=64   ANN@10 0.9545  R

In [12]:
# WHAT: Repeats the dev timing with FAISS limited to 1 thread (smaller grid), saved with the tag "dev_1thread".
# WHY:  Colab has 2 CPU cores. For one query at a time, FAISS's multi-threading overhead can make it look slower
#       than it is; the single-thread numbers show the per-query cost without that overhead.
!python -m eval.benchmark_faiss --splits dev --threads 1 --hnsw-m 32 --ef-search 32 64 128 --tag dev_1thread

Loading weights: 100% 199/199 [00:00<00:00, 4684.37it/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
28541 chunk vectors, dim 384 | faiss 1.15.1 | 2 CPUs, FAISS threads 1
FAISS IndexFlatIP built in 0.03 s (28541 vectors, dim 384)
FAISS IndexHNSWFlat(M=32, efConstruction=200, efSearch=128) built in 15.26 s (28541 vectors, dim 384)
Loading cached query embeddings: /content/drive/MyDrive/research-engine/cache/qemb_BAAI_bge-small-en-v1.5_dev_887_c304fa793f01.npy

=== dev: 887 questions
  HNSW M=32  ef=32   ANN@10 0.9023  R@5 0.1105  0.174 ms/q
  HNSW M=32  ef=64   ANN@10 0.9545  R@5 0.1066  0.260 ms/q
  HNSW M=32  ef=128  ANN@10 0.9835  R@5 0.1087  0.415 ms/q
  Flat vs numpy top-10: identical order 0.9989, identical set 1.0000, max score gap where different 0.00e+00
  Flat vs numpy top-50: identical order 0.9944, identical set 1.0000, max score gap where different 1.19e-07
  Flat vs numpy top-100: identical order 0.9775, iden

In [13]:
# WHAT: Shows both benchmark reports as tables.
# WHY:  To read the results (and paste them into the results log).
from IPython.display import Markdown, display
for f in ["results/faiss_benchmark.md", "results/faiss_benchmark_dev_1thread.md"]:
    display(Markdown(open(f).read()))

# FAISS vs numpy, corpus-wide dense search (28541 chunks, dim 384)

faiss 1.15.1, numpy 2.1.3, 2 CPUs, FAISS threads 2. Latency = search only (query vectors pre-encoded), one query at a time; batch = one call for all queries, per query.

| index | build s | size MB |
|---|---|---|
| flat | 0.03 | 43.8 |
| numpy | 0.0 | 43.8 |
| hnsw_M16 | 9.29 | 48.0 |
| hnsw_M32 | 10.41 | 51.6 |

## dev (n=887)

- Flat vs numpy top-10: identical order 0.9977, identical set 1.0000, max score gap where different 2.4e-07
- Flat vs numpy top-50: identical order 0.9853, identical set 1.0000, max score gap where different 4.2e-07
- Flat vs numpy top-100: identical order 0.9504, identical set 0.9989, max score gap where different 5.4e-07
- Gold metrics equal: True -> **exact**
- Day 3 dev corpus dense check: ev_recall@5 0.1098 vs 0.1098, ev_recall@10 0.1550 vs 0.1550, hit@1 0.0620 vs 0.0620

| system | ANN@10 | ANN@50 | R@5 | R@10 | R@50 | hit@1 | median ms | p95 ms | batch ms/q |
|---|---|---|---|---|---|---|---|---|---|
| numpy argsort (current) | 1.0000 | 1.0000 | 0.1098 | 0.1550 | 0.2482 | 0.0620 | 3.152 | 4.308 |  |
| numpy argpartition | 1.0000 | 1.0000 |  |  |  |  | 2.325 | 3.815 |  |
| FAISS IndexFlatIP | 1.0000 | 1.0000 | 0.1098 | 0.1550 | 0.2482 | 0.0620 | 6.304 | 10.543 | 0.250 |
| FAISS HNSW M=16 efSearch=16 | 0.7823 | 0.6276 | 0.0995 | 0.1337 | 0.2037 | 0.0519 | 0.099 | 0.148 | 0.058 |
| FAISS HNSW M=16 efSearch=32 | 0.8786 | 0.7653 | 0.1021 | 0.1401 | 0.2184 | 0.0541 | 0.138 | 0.199 | 0.085 |
| FAISS HNSW M=16 efSearch=64 | 0.9408 | 0.8789 | 0.1047 | 0.1474 | 0.2331 | 0.0575 | 0.214 | 0.298 | 0.134 |
| FAISS HNSW M=16 efSearch=128 | 0.9752 | 0.9461 | 0.1070 | 0.1516 | 0.2411 | 0.0586 | 0.354 | 0.541 | 0.222 |
| FAISS HNSW M=16 efSearch=256 | 0.9911 | 0.9798 | 0.1087 | 0.1539 | 0.2471 | 0.0609 | 0.593 | 0.739 | 0.440 |
| FAISS HNSW M=32 efSearch=16 | 0.8101 | 0.6678 | 0.0983 | 0.1354 | 0.2155 | 0.0507 | 0.119 | 0.165 | 0.071 |
| FAISS HNSW M=32 efSearch=32 | 0.9023 | 0.8046 | 0.1105 | 0.1462 | 0.2297 | 0.0575 | 0.167 | 0.242 | 0.104 |
| FAISS HNSW M=32 efSearch=64 | 0.9545 | 0.9021 | 0.1066 | 0.1488 | 0.2389 | 0.0586 | 0.276 | 0.390 | 0.157 |
| FAISS HNSW M=32 efSearch=128 | 0.9835 | 0.9608 | 0.1087 | 0.1520 | 0.2477 | 0.0609 | 0.395 | 0.499 | 0.269 |
| FAISS HNSW M=32 efSearch=256 | 0.9940 | 0.9857 | 0.1087 | 0.1539 | 0.2471 | 0.0609 | 0.717 | 0.946 | 0.511 |

## test (n=1309)

- Flat vs numpy top-10: identical order 0.9985, identical set 1.0000, max score gap where different 3.6e-07
- Flat vs numpy top-50: identical order 0.9763, identical set 1.0000, max score gap where different 3.6e-07
- Flat vs numpy top-100: identical order 0.9458, identical set 0.9992, max score gap where different 4.8e-07
- Gold metrics equal: True -> **exact**

| system | ANN@10 | ANN@50 | R@5 | R@10 | R@50 | hit@1 | median ms | p95 ms | batch ms/q |
|---|---|---|---|---|---|---|---|---|---|
| numpy argsort (current) | 1.0000 | 1.0000 | 0.1246 | 0.1644 | 0.2645 | 0.0749 | 3.004 | 3.857 |  |
| numpy argpartition | 1.0000 | 1.0000 |  |  |  |  | 2.327 | 2.906 |  |
| FAISS IndexFlatIP | 1.0000 | 1.0000 | 0.1246 | 0.1644 | 0.2645 | 0.0749 | 6.309 | 9.706 | 0.232 |
| FAISS HNSW M=16 efSearch=16 | 0.7759 | 0.6218 | 0.1052 | 0.1383 | 0.2243 | 0.0619 | 0.104 | 0.178 | 0.081 |
| FAISS HNSW M=16 efSearch=32 | 0.8769 | 0.7649 | 0.1139 | 0.1487 | 0.2370 | 0.0665 | 0.140 | 0.193 | 0.092 |
| FAISS HNSW M=16 efSearch=64 | 0.9416 | 0.8751 | 0.1179 | 0.1564 | 0.2475 | 0.0703 | 0.210 | 0.282 | 0.134 |
| FAISS HNSW M=16 efSearch=128 | 0.9762 | 0.9438 | 0.1254 | 0.1639 | 0.2596 | 0.0749 | 0.349 | 0.509 | 0.265 |
| FAISS HNSW M=16 efSearch=256 | 0.9904 | 0.9777 | 0.1239 | 0.1636 | 0.2613 | 0.0749 | 0.605 | 0.814 | 0.422 |
| FAISS HNSW M=32 efSearch=16 | 0.8086 | 0.6662 | 0.1100 | 0.1468 | 0.2298 | 0.0611 | 0.120 | 0.169 | 0.074 |
| FAISS HNSW M=32 efSearch=32 | 0.8963 | 0.8004 | 0.1151 | 0.1526 | 0.2418 | 0.0680 | 0.166 | 0.226 | 0.103 |
| FAISS HNSW M=32 efSearch=64 | 0.9532 | 0.9014 | 0.1197 | 0.1585 | 0.2516 | 0.0733 | 0.244 | 0.322 | 0.164 |
| FAISS HNSW M=32 efSearch=128 | 0.9820 | 0.9598 | 0.1233 | 0.1602 | 0.2582 | 0.0733 | 0.407 | 0.543 | 0.287 |
| FAISS HNSW M=32 efSearch=256 | 0.9931 | 0.9857 | 0.1254 | 0.1636 | 0.2632 | 0.0741 | 0.706 | 0.889 | 0.480 |

DenseIndex.search(ids=None) numpy vs flat on 200 dev questions, top-50: identical 0.9850


# FAISS vs numpy, corpus-wide dense search (28541 chunks, dim 384)

faiss 1.15.1, numpy 2.1.3, 2 CPUs, FAISS threads 1. Latency = search only (query vectors pre-encoded), one query at a time; batch = one call for all queries, per query.

| index | build s | size MB |
|---|---|---|
| flat | 0.03 | 43.8 |
| numpy | 0.0 | 43.8 |
| hnsw_M32 | 15.26 | 51.6 |

## dev (n=887)

- Flat vs numpy top-10: identical order 0.9989, identical set 1.0000, max score gap where different 0.0e+00
- Flat vs numpy top-50: identical order 0.9944, identical set 1.0000, max score gap where different 1.2e-07
- Flat vs numpy top-100: identical order 0.9775, identical set 0.9989, max score gap where different 1.2e-07
- Gold metrics equal: True -> **exact**
- Day 3 dev corpus dense check: ev_recall@5 0.1098 vs 0.1098, ev_recall@10 0.1550 vs 0.1550, hit@1 0.0620 vs 0.0620

| system | ANN@10 | ANN@50 | R@5 | R@10 | R@50 | hit@1 | median ms | p95 ms | batch ms/q |
|---|---|---|---|---|---|---|---|---|---|
| numpy argsort (current) | 1.0000 | 1.0000 | 0.1098 | 0.1550 | 0.2482 | 0.0620 | 2.993 | 4.271 |  |
| numpy argpartition | 1.0000 | 1.0000 |  |  |  |  | 2.245 | 3.305 |  |
| FAISS IndexFlatIP | 1.0000 | 1.0000 | 0.1098 | 0.1550 | 0.2482 | 0.0620 | 2.593 | 3.411 | 0.262 |
| FAISS HNSW M=32 efSearch=32 | 0.9023 | 0.8046 | 0.1105 | 0.1462 | 0.2297 | 0.0575 | 0.174 | 0.242 | 0.163 |
| FAISS HNSW M=32 efSearch=64 | 0.9545 | 0.9021 | 0.1066 | 0.1488 | 0.2389 | 0.0586 | 0.260 | 0.336 | 0.225 |
| FAISS HNSW M=32 efSearch=128 | 0.9835 | 0.9608 | 0.1087 | 0.1520 | 0.2477 | 0.0609 | 0.415 | 0.533 | 0.387 |

DenseIndex.search(ids=None) numpy vs flat on 200 dev questions, top-50: identical 0.9900


## Step 4 — corpus-wide retrieval through the whole pipeline

In [14]:
# WHAT: Runs the full pipeline with paper_id=None: BM25 over all 28.5k chunks + dense via FAISS flat -> RRF ->
#       reranker -> top 5, and prints which paper each hit comes from.
# WHY:  This is the call the cross-paper mode will use. Hits from several papers show that the corpus-wide
#       path works end to end. (corpus_index="flat" is the default in RetrievalConfig.)
from retrieval import retrieve
for h in retrieve("Which datasets are used for sentiment classification?", paper_id=None, k=5):
    print(f"{h['rank']}. {h['score']:.3f}  {h['paper_id']:<12} {h['section']:<25.25} {h['text'][:90]!r}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.npy
FAISS IndexFlatIP built in 0.03 s (28541 vectors, dim 384)


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1. 1.000  1908.09590   Experiments ::: Sentiment '[Experiments ::: Sentiment Classification ::: Datasets and Evaluation] We use the three wi'
2. 0.997  1812.06705   Datasets                  '[Datasets] Six benchmark classification datasets are listed in table 1 . Following Kim [re'
3. 0.997  1611.01884   Datasets                  '[Datasets] We evaluate our model on various benchmarks. Stanford Sentiment Treebank (SST) '
4. 0.994  1810.07091   Datasets and External Res '[Datasets and External Resources] We use the the datasets provided by Zhang et al. zhang20'
5. 0.990  1908.11860   Related Works ::: Archite '[Related Works ::: Architecture Improvements for Aspect-Target Sentiment Classification] T'


## Results (2026-10-15, T4 runtime, 2 CPUs, faiss 1.15.1)
* **Gate:** test final R@5 **0.795** (R@10 0.941, hit@1 0.522, MRR 0.677) -> REPRODUCED. FAISS is only used for
  corpus-wide search, so the frozen within-paper pipeline is unchanged.
* **Flat vs numpy (exact):** evidence recall identical on all 2,196 dev+test questions; dev reproduces Day 3
  corpus dense exactly (R@5 0.1098). Top-10/50 chunk sets identical for every question; the only order
  differences are ties whose scores differ by < 6e-7 (float rounding).
* **Speed (one query, search only):** numpy 3.0 ms; FAISS flat 6.3 ms with 2 threads (thread start-up cost)
  but 2.6 ms with 1 thread; FAISS flat batched 0.25 ms/query. Query encoding alone is ~10 ms.
* **HNSW (M=32):** efSearch=128 finds 98% of the true top-10 at 0.41 ms/query (7.4x faster than numpy),
  R@5 0.1233 vs exact 0.1246 (test). efSearch=64: 95%, 0.24 ms. efSearch=256: 99%, 0.71 ms. Build ~10 s, 52 MB.
* **Decision:** cross-paper mode uses FAISS flat (exact). At 28.5k chunks the search is a few ms next to
  ~260 ms reranking and seconds of LLM time, so HNSW's speed-up isn't worth losing 2% of neighbours.
  HNSW stays available (`corpus_index="hnsw"`) for a larger corpus.
* **Corpus-wide demo:** `retrieve(..., paper_id=None)` returns 5 relevant chunks from 5 different papers.